In [8]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch_geometric 
import torch_geometric.nn as gnn
from torch_geometric.loader import DataLoader as gDataLoader
from torch_geometric.data import Dataset as gDataset
from torch_geometric.data import Data
from torch_geometric.datasets import QM9
import rdkit
from rdkit.Chem import MolFromSmiles as get_mol
import os
import numpy as np 
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available else "cpu")

In [9]:
def get_ei(smi) : 
    mol = rdkit.Chem.MolFromSmiles(smi) 
    n_atoms = mol.GetNumAtoms() 
    ei = []
    for bond in mol.GetBonds() :
        b = bond.GetBeginAtomIdx() 
        e = bond.GetEndAtomIdx() 
        ei.append([b,e])
    for bond in mol.GetBonds() :
        b = bond.GetBeginAtomIdx() 
        e = bond.GetEndAtomIdx() 
        ei.append([e, b])
   
    ei.append([n_atoms, n_atoms - 1])
    return torch.tensor(ei).T

def pad_ei(ei, max_ei) : 
    ei = ei.T.tolist() 
    end = max(max(ei))
    for i in range(max_ei - len(ei)) : 
        ei.append([end, end + i + 1])
    return torch.tensor(ei).T

def get_vocab(smi_list) :
    dic = {"<end>":0, "<pad>":1}
    for smi in smi_list :
        mol = rdkit.Chem.MolFromSmiles(smi) 
        for atom in mol.GetAtoms() : 
            symbol = atom.GetSymbol() 
            if symbol not in dic : 
                dic[symbol] = len(dic) 
    return dic 


def get_nf(smi, vocab) : 
    mol = rdkit.Chem.MolFromSmiles(smi) 
    atom_list = [vocab[atom.GetSymbol()] for atom in mol.GetAtoms()]
    atom_list.append(0)
    return atom_list

def pad_nf(nf, max_nf) : 
    nf = nf + [1] * (max_nf - len(nf))
    return torch.tensor(nf)

        
def get_atom_mat(smi, vocab) : 
    mol = get_mol(smi) 
    mat = rdkit.Chem.rdmolops.GetAdjacencyMatrix(mol)

    for i, atom in enumerate(mol.GetAtoms()) : 
        neighbor = list(atom.GetNeighbors())
        symbol = [(n.GetSymbol(), n.GetIdx()) for n in neighbor]
        for s in symbol : 
            mat[i][s[1]] = vocab[s[0]]
    return mat 
        
def get_bond_mat(smi) : 
    mol = rdkit.Chem.MolFromSmiles(smi) 
    mat = rdkit.Chem.rdmolops.GetAdjacencyMatrix(mol, useBO=True, emptyVal=0) 
    return mat

def pad_mat(mat, max_len) :
   cur_size = mat.shape[0] 
   out = np.zeros((max_len, max_len))
   out[:cur_size, :cur_size] = mat 
   return torch.tensor(out).view(-1)

In [10]:
class MyData(Data) : 
    def __cat_dim__(self, key, value, *args, **kwargs) : 
        if key == 'nf' or key == 'ei' : 
            return None 
        return super().__cat_dim__(key, value, *args, **kwargs) 

In [11]:
class MyDataset(gDataset) : 
    def __init__(self, root, filename, transform=None, pre_transform=None) : 
        self.filename = filename 
        super(MyDataset, self).__init__(root, transform, pre_transform)
        self.vocab = get_vocab(self.smi_list)
        
    @property 
    def raw_file_names(self) : 
        return self.filename
    
    @property
    def processed_file_names(self) : 
        with open(self.raw_paths[0], 'r') as f : 
            self.smi_list = f.readlines()
            self.smi_list = [s.strip() for s in self.smi_list if len(s) < 30]
        return [f'data_{i}.pt' for i in range(len(self.smi_list))]
    
    def download(self) : pass 

    def process(self) : 
        with open(self.raw_paths[0], 'r') as f : 
            self.smi_list = f.readlines()
            self.smi_list = [s.strip() for s in self.smi_list]
            
            ei_list = [get_ei(s) for s in self.smi_list]
            max_ei = max(ei_list, key =lambda x:x.size(1)).size(1)
            ei_list = [pad_ei(ei, max_ei) for ei in ei_list]

            vocab = get_vocab(self.smi_list)
            nf_list = [get_nf(s, vocab) for s in self.smi_list]
            max_nf = len(max(nf_list))
            nf_list = [pad_nf(nf, max_ei) for nf in nf_list]
        
        for i, smi in tqdm(enumerate(self.smi_list)) : 
            data = MyData(x=nf_list[i],
                        edge_index=ei_list[i],
                        nf=nf_list[i],
                        ei=ei_list[i],)
            torch.save(data, os.path.join(self.processed_dir, f'data_{i}.pt'))
    
    def len(self) : 
        return len(self.smi_list) 
    
    def get(self, idx) : 
        data = torch.load(os.path.join(self.processed_dir, f'data_{idx}.pt'))
        return data
        

In [12]:
dataset = MyDataset(root='data', filename=['25_MOSES_SMILES.txt'])
train_loader = gDataLoader(dataset, batch_size=128)
vocab = dataset.vocab

In [46]:
print(dataset)

MyDataset(4260)


In [277]:
# class GVAE(nn.Module) : 
#     def __init__(self, d_model, d_latent, max_len) : 
#         super(GVAE, self).__init__()

#         self.embedding = nn.Embedding(len(vocab), d_model) 
#         self.gcn1 = gnn.GCNConv(d_model, d_model) 
#         self.drop1 = nn.Dropout()


#         self.gcn2 = gnn.GCNConv(d_model, d_model) 

#         self.mu = nn.Linear(d_model, d_latent)
#         self.sigma = nn.Linear(d_model, d_latent)

#         self.dec1 = nn.Sequential(
#             nn.Linear(d_latent, d_model),
#             nn.LayerNorm(d_model),
#             nn.ReLU()
#         )

#         self.upsize = nn.Linear(d_latent, max_len * max_len)

#         self.proj = nn.Sequential(
#             nn.Linear(1, d_model),
#             nn.LayerNorm(d_model),
#             nn.ReLU(),
#             nn.Linear(d_model, len(vocab))
#         )
#     def get_z(self, mu, sigma) : 
#         eps = torch.rand_like(sigma).to(device) 
#         z = mu + torch.exp(0.5 * sigma) * eps 
#         return z 
    
#     def inference(self, z) :
#         z = self.upsize(z)
#         z = z.unsqueeze(-1)

#         out = self.proj(z) 
#         out = F.log_softmax(out, dim=-1)
#         return out 
#     def forward(self, x) : 
#         nf, ei, batch = x.x, x.edge_index, x.batch 

#         nf = self.embedding(nf) 
#         nf = self.drop1(self.gcn1(nf, ei).relu())
#         # print(f'after gcn1: {nf}')
#         nf = self.gcn2(nf, ei) 
#         # print(f'after gcn2: {nf}')

#         pool = gnn.global_add_pool(nf, batch)
#         # print(f'after pool: {pool}')

#         mu, sigma = self.mu(pool), self.sigma(pool) 
#         # print(f'after mu: {mu}')
#         # print(f'after sigma: {sigma}')

#         z = self.get_z(mu, sigma) 
#         # print(f'after z: {z}')

#         z = self.upsize(z)
#         # print(f'after upsize: {z}')

#         z = z.unsqueeze(-1)

#         out = self.proj(z) 
#         out = F.log_softmax(out, dim=-1)
#         # print(f'after out: {out}')

#         return out, mu, sigma 

In [25]:
class GVAE(nn.Module) : 
    def __init__(self, d_model, d_latent) : 
        super(GVAE, self).__init__()

        self.embed = nn.Embedding(len(vocab), d_model)

        self.gcn1 = gnn.GCNConv(d_model, d_model) 
        self.norm1 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout()
        
        self.mu = gnn.GCNConv(d_model, d_latent)
        self.sigma = gnn.GCNConv(d_model, d_latent) 

        self.dec = nn.Sequential(
            nn.Linear(d_latent, d_model),
            nn.ReLU(),
            nn.LayerNorm(d_model),
            nn.Linear(d_model, len(vocab))
        )
    def get_z(self, mu, sigma) : 
        eps = torch.rand_like(sigma).to(device) 
        z = mu + torch.exp(0.5 * sigma) * eps 
        return z 
    
    def inference(self, z)  :
        out = self.dec(z) 
        out = F.log_softmax(out, dim=-1)
        return out
    
    def forward(self, x) : 
        nf, ei = x.x, x.edge_index 

        nf = self.embed(nf)
        nf = self.drop1(self.norm1(self.gcn1(nf, ei)).relu())

        mu = self.mu(nf, ei)
        sigma = self.sigma(nf, ei)
        z = self.get_z(mu, sigma)
        out = self.inference(z)
        return out, mu, sigma

In [26]:
model = GVAE(256, 128).to(device)
optim = torch.optim.Adam(model.parameters(), lr = 0.0003, weight_decay=1e-6)
def loss_fn(pred, tgt, mu, sigma, beta) :
    reconstruction_loss = F.nll_loss(pred.reshape(-1, len(vocab)), tgt.reshape(-1))
    kl_loss = -0.5 * torch.sum(1 + sigma - mu.pow(2) - sigma.exp()).mean() / 64
    return  reconstruction_loss + kl_loss * beta

In [27]:
for src in train_loader : 
    src = src.to(device)
    tgt = src.clone().detach().x

    pred, mu, sigma = model(src)
    break 

torch.Size([5248])


In [43]:
for epoch in range(1) : 
    train_loss = 0
    for src in train_loader : 
        src = src.to(device) 
        tgt = src.clone().detach().x

        pred, mu, sigma = model(src)
        loss = loss_fn(pred, tgt, mu, sigma, 0)
        optim.zero_grad(), loss.backward(), optim.step()
        train_loss += loss.item()
    print(f'epoch: {epoch}, loss: {train_loss / len(train_loader)}')


epoch: 0, loss: 0.06643068133031621


In [45]:
model.eval()
z = torch.randn(41, 128).to(device)
pred = model.inference(z).squeeze(0)

_, idx = torch.topk(pred, 1, dim=-1)
# idx = idx.squeeze(-1)
idx
# idx.reshape(max_len, max_len)

tensor([[2],
        [3],
        [5],
        [6],
        [8],
        [7],
        [2],
        [2],
        [2],
        [0],
        [1],
        [0],
        [1],
        [5],
        [4],
        [1],
        [5],
        [3],
        [2],
        [0],
        [1],
        [6],
        [4],
        [0],
        [0],
        [1],
        [1],
        [0],
        [3],
        [7],
        [7],
        [4],
        [1],
        [6],
        [2],
        [5],
        [1],
        [7],
        [6],
        [7],
        [4]], device='cuda:0')

In [30]:
vocab

{'<end>': 0,
 '<pad>': 1,
 'N': 2,
 'C': 3,
 'O': 4,
 'Br': 5,
 'S': 6,
 'F': 7,
 'Cl': 8}

In [150]:
with open('data/raw/25_MOSES_SMILES.txt', 'r') as f : 
    smi_list = f.readlines()
    smi_list = [s.strip() for s in smi_list]
    
    ei_list = [get_ei(s) for s in smi_list]
    max_ei = max(ei_list, key =lambda x:x.size(1)).size(1)
    ei_list = [pad_ei(ei, max_ei) for ei in ei_list]

    vocab = get_vocab(smi_list)
    nf_list = [get_nf(s, vocab) for s in smi_list]
    max_nf = len(max(nf_list))
    nf_list = [pad_nf(nf, max_ei) for nf in nf_list]
    

    print(ei_list[3])

tensor([[ 0,  1,  2,  2,  4,  5,  5,  7,  8,  9, 10, 11, 11,  1,  2,  3,  4,  5,
          6,  7,  8,  9, 10, 11,  1,  7, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12,
         12, 12, 12, 12, 12],
        [ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11,  1,  7,  0,  1,  2,  2,  4,
          5,  5,  7,  8,  9, 10, 11, 11, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21,
         22, 23, 24, 25, 26]])


In [151]:
nf_list[0], ei_list[0]

(tensor([2, 3, 4, 3, 3, 3, 5, 3, 3, 5, 3, 4, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 tensor([[ 0,  1,  1,  3,  4,  5,  5,  7,  8,  8, 10, 10,  1,  2,  3,  4,  5,  6,
           7,  8,  9, 10, 11,  3, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12,
          12, 12, 12, 12, 12],
         [ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11,  3,  0,  1,  1,  3,  4,  5,
           5,  7,  8,  8, 10, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23,
          24, 25, 26, 27, 28]]))

In [153]:
model = gnn.GCNConv(100, 100) 
embed=  nn.Embedding(len(vocab), 100)
nf = embed(nf_list[0])
out = model(nf, ei_list[0])
out

tensor([[ 0.2641,  0.3409,  0.2277,  ..., -0.7346, -0.2004, -0.2751],
        [-0.3042,  0.2332,  0.5564,  ..., -1.0456, -1.1099, -0.5369],
        [-0.6943, -0.0111,  0.5591,  ..., -0.7442, -1.3693, -0.4842],
        ...,
        [ 1.7395, -0.7128,  0.2517,  ..., -0.5566,  0.8187,  1.4301],
        [ 1.7395, -0.7128,  0.2517,  ..., -0.5566,  0.8187,  1.4301],
        [ 1.7395, -0.7128,  0.2517,  ..., -0.5566,  0.8187,  1.4301]],
       grad_fn=<AddBackward0>)